# Finetuning Training-State Plots

Reusable utilities to load a Hugging Face Trainer state JSON and plot finetuning metrics for one or multiple runs.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
def load_trainer_log(json_path):
    """Load trainer_state.json and return (trainer_state, dataframe)."""
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f'JSON file not found: {path}')

    with path.open('r', encoding='utf-8') as f:
        trainer_state = json.load(f)

    log_history = trainer_state.get('log_history', [])
    if not log_history:
        raise ValueError(f'No log_history found in JSON: {path}')

    df = pd.DataFrame(log_history)
    if 'step' in df.columns:
        df = df.sort_values('step')
    df = df.reset_index(drop=True)

    for col in ['step', 'epoch', 'loss', 'learning_rate', 'grad_norm']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return trainer_state, df


def plot_finetuning_log(json_path, metrics=None, show_epoch_plots=True, show_summary=True):
    """Plot metrics from a trainer_state JSON path."""
    trainer_state, df = load_trainer_log(json_path)

    if metrics is None:
        metrics = ['loss', 'learning_rate', 'grad_norm']
    metrics = [m for m in metrics if m in df.columns]

    if not metrics:
        raise ValueError('None of the requested metrics are present in log_history.')

    print(f'Loaded: {json_path}')
    print(f"Global step: {trainer_state.get('global_step')} / Max steps: {trainer_state.get('max_steps')}")
    print(f"Epoch: {trainer_state.get('epoch')}")
    print(f'Rows in log_history: {len(df)}')

    display(df.head())
    display(df.tail())

    # Plot by step
    n = len(metrics)
    fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        sns.lineplot(data=df, x='step', y=metric, ax=ax, linewidth=1.5)
        ax.set_title(f'{metric} over training steps')
        ax.set_xlabel('step')
        ax.set_ylabel(metric)

    plt.tight_layout()
    plt.show()

    # Plot by epoch
    if show_epoch_plots and 'epoch' in df.columns:
        fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n), sharex=True)
        if n == 1:
            axes = [axes]

        for ax, metric in zip(axes, metrics):
            sns.lineplot(data=df, x='epoch', y=metric, ax=ax, linewidth=1.5)
            ax.set_title(f'{metric} over epochs')
            ax.set_xlabel('epoch')
            ax.set_ylabel(metric)

        plt.tight_layout()
        plt.show()

    if show_summary:
        summary = df[metrics].describe(percentiles=[0.1, 0.5, 0.9]).T
        display(summary)

    return trainer_state, df

In [ ]:
# Example usage: duplicate this cell for each model
# trainer_state, df = plot_finetuning_log(r"C:\path\to\trainer_state.json")